In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 70)


# Просмотр данных

Таблицы dashboard_risk_prediction и dashboard_risk_prediction_distributed

CREATE TABLE IF NOT EXISTS zpe_aso_dva_urvk.dashboard_risk_prediction ON CLUSTER superset (
    bp_number Int64,
    bp_name String,
    risk_name String,
    report_month DateTime,
    residual_risk_zone String,
    present_risk_zone String,
    risk_event_probability Float64,
    event_pred String,
    prob_level_event String,
    prob_desc_event String,
    prob_mean_event Float64,
    n_sources_triggered Int64,
    n_unique_measures Int64,
    source_label_1 String,
    probability_1 Float64,
    prediction_1 String,
    prob_level_1 String,
    prob_desc_1 String,
    prob_mean_1 Float64,
    src_events_12m_1 Int64,
    avg_monthly_events_1 Float64,
    source_label_2 String,
    probability_2 Float64,
    prediction_2 String,
    prob_level_2 String,
    prob_desc_2 String,
    prob_mean_2 Float64,
    src_events_12m_2 Int64,
    avg_monthly_events_2 Float64,
    source_label_3 String,
    probability_3 Float64,
    prediction_3 String,
    prob_level_3 String,
    prob_desc_3 String,
    prob_mean_3 Float64,
    src_events_12m_3 Int64,
    avg_monthly_events_3 Float64,
    source_label_4 String,
    probability_4 Float64,
    prediction_4 String,
    prob_level_4 String,
    prob_desc_4 String,
    prob_mean_4 Float64,
    src_events_12m_4 Int64,
    avg_monthly_events_4 Float64,
    risk_key String
)
ENGINE = ReplicatedMergeTree('/clickhouse/tables/{uuid}/{shard}', '{replica}')
ORDER BY risk_key;

CREATE TABLE IF NOT EXISTS zpe_aso_dva_urvk.dashboard_risk_prediction_distributed ON CLUSTER superset AS zpe_aso_dva_urvk.dashboard_risk_prediction
ENGINE = Distributed('superset','zpe_aso_dva_urvk', 'dashboard_risk_prediction', rand());


In [2]:
dashboard_risk_prediction = pd.read_csv('dashboard_risk_prediction.csv', sep = ';')
dashboard_risk_prediction.fillna(0, inplace = True)
print(dashboard_risk_prediction.shape)
dashboard_risk_prediction['report_month'] = pd.to_datetime(dashboard_risk_prediction['report_month'], format = '%Y-%m')
dashboard_risk_prediction.head(1)

(62, 46)


,bp_number,bp_name,risk_name,report_month,residual_risk_zone,present_risk_zone,risk_event_probability,event_pred,prob_level_event,prob_desc_event,prob_mean_event,n_sources_triggered,n_unique_measures,source_label_1,probability_1,prediction_1,prob_level_1,prob_desc_1,prob_mean_1,src_events_12m_1,avg_monthly_events_1,source_label_2,probability_2,prediction_2,prob_level_2,prob_desc_2,prob_mean_2,src_events_12m_2,avg_monthly_events_2,source_label_3,probability_3,prediction_3,prob_level_3,prob_desc_3,prob_mean_3,src_events_12m_3,avg_monthly_events_3,source_label_4,probability_4,prediction_4,prob_level_4,prob_desc_4,prob_mean_4,src_events_12m_4,avg_monthly_events_4,risk_key
0,30,0030: название бизнес-процесса.,РC-0104,2025-11-01,Красная,Красная,0.48,Не сработает,Норма,В пределах нормы,0.52,0,29,Источник 1,0.58,Не сработает,Не сработает,Нет сигнала,0.0,7,1.75,Источник 2,0.44,Не сработает,Не сработает,Нет сигнала,0.0,0,0.0,Источник 3,0.38,Не сработает,Не сработает,Нет сигнала,0.0,2,1.0,Источник 4,0.0,Не сработает,Не сработает,Нет сигнала,0.0,0,0.0,30|РC-0104


In [3]:
print(dashboard_risk_prediction['bp_number'].nunique())
print(dashboard_risk_prediction['risk_name'].nunique())

13
62


In [4]:
dashboard_risk_prediction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 46 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   bp_number               62 non-null     int64         
 1   bp_name                 62 non-null     object        
 2   risk_name               62 non-null     object        
 3   report_month            62 non-null     datetime64[ns]
 4   residual_risk_zone      62 non-null     object        
 5   present_risk_zone       62 non-null     object        
 6   risk_event_probability  62 non-null     float64       
 7   event_pred              62 non-null     object        
 8   prob_level_event        62 non-null     object        
 9   prob_desc_event         62 non-null     object        
 10  prob_mean_event         62 non-null     float64       
 11  n_sources_triggered     62 non-null     int64         
 12  n_unique_measures       62 non-null     int64       

CREATE TABLE IF NOT EXISTS zpe_aso_dva_urvk.dashboard_measures ON CLUSTER superset (
    bp_number Int64,
    risk_name String,
    source_number Int64,
    n_applied Int64,
    n_effective Int64,
    measure_type String,
    report_month DateTime,
    source_label String,
    effectiveness Float64,
    effectiveness_level String,
    dominant_consequence_kind String,
    dominant_consequence_type String,
    risk_key String,
)
ENGINE = ReplicatedMergeTree('/clickhouse/tables/{uuid}/{shard}', '{replica}')
ORDER BY risk_key;

CREATE TABLE IF NOT EXISTS zpe_aso_dva_urvk.dashboard_measures_distributed ON CLUSTER superset AS zpe_aso_dva_urvk.dashboard_measures
ENGINE = Distributed('superset','zpe_aso_dva_urvk', 'dashboard_measures', rand());



In [5]:
dashboard_measures = pd.read_csv('dashboard_measures.csv', sep = ';')
dashboard_measures['report_month'] = pd.to_datetime(dashboard_measures['report_month'], format = '%Y-%m')
dashboard_measures = dashboard_measures[dashboard_measures['risk_key'].isin(dashboard_risk_prediction['risk_key'])]
print(dashboard_measures.shape)
dashboard_measures.head(1)

(1984, 13)


,bp_number,risk_name,risk_key,source_number,source_label,measure_type,n_applied,n_effective,effectiveness,effectiveness_level,dominant_consequence_kind,dominant_consequence_type,report_month
0,30,РC-0104,30|РC-0104,1,Источник 1,Мера 4,6,2,0.33,Неэффективна,Фактическое,Деятельность,2025-11-01


In [6]:
print(dashboard_measures['bp_number'].nunique())
print(dashboard_measures['risk_name'].nunique())

13
62


In [7]:
dashboard_measures.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1984 entries, 0 to 1983
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   bp_number                  1984 non-null   int64         
 1   risk_name                  1984 non-null   object        
 2   risk_key                   1984 non-null   object        
 3   source_number              1984 non-null   int64         
 4   source_label               1984 non-null   object        
 5   measure_type               1984 non-null   object        
 6   n_applied                  1984 non-null   int64         
 7   n_effective                1984 non-null   int64         
 8   effectiveness              1984 non-null   float64       
 9   effectiveness_level        1984 non-null   object        
 10  dominant_consequence_kind  1984 non-null   object        
 11  dominant_consequence_type  1984 non-null   object        
 12  report

CREATE TABLE IF NOT EXISTS zpe_aso_dva_urvk.dashboard_timeline ON CLUSTER superset (
bp_number Int64,
risk_name String,
year_month DateTime,
source_number Int64,
source_label String,
event_count Int64,
loss_sum  Float64,
n_factual Int64,
n_possible Int64,
n_avoided Int64,
n_cons_activity Int64,
n_cons_reputation Int64,
n_cons_financial Int64,
risk_key String,
)
ENGINE = ReplicatedMergeTree('/clickhouse/tables/{uuid}/{shard}', '{replica}')
ORDER BY risk_key;

CREATE TABLE IF NOT EXISTS zpe_aso_dva_urvk.dashboard_timeline_distributed ON CLUSTER superset AS zpe_aso_dva_urvk. dashboard_timeline
ENGINE = Distributed('superset','zpe_aso_dva_urvk', 'dashboard_timeline', rand());


In [8]:
dashboard_timeline = pd.read_csv('dashboard_timeline.csv', sep = ';')
dashboard_timeline['year_month'] = pd.to_datetime(dashboard_timeline['year_month'], format = '%Y-%m')
cols = ['n_factual', 'n_possible', 'n_avoided', 'n_cons_activity', 'n_cons_reputation', 'n_cons_financial']
dashboard_timeline[cols] = dashboard_timeline[cols].astype(int)
print(dashboard_timeline.shape)
dashboard_timeline.head()

(11160, 14)


,bp_number,risk_name,year_month,source_number,source_label,event_count,loss_sum,n_factual,n_possible,n_avoided,n_cons_activity,n_cons_reputation,n_cons_financial,risk_key
0,2,РC-0004,2022-11-01,0,Все источники,3,0.0,2,0,1,3,0,0,2|РC-0004
1,2,РC-0004,2022-12-01,0,Все источники,0,0.0,0,0,0,0,0,0,2|РC-0004
2,2,РC-0004,2023-01-01,0,Все источники,0,0.0,0,0,0,0,0,0,2|РC-0004
3,2,РC-0004,2023-02-01,0,Все источники,0,0.0,0,0,0,0,0,0,2|РC-0004
4,2,РC-0004,2023-03-01,0,Все источники,2,0.0,0,0,2,0,2,0,2|РC-0004


In [9]:
dashboard_timeline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11160 entries, 0 to 11159
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   bp_number          11160 non-null  int64         
 1   risk_name          11160 non-null  object        
 2   year_month         11160 non-null  datetime64[ns]
 3   source_number      11160 non-null  int64         
 4   source_label       11160 non-null  object        
 5   event_count        11160 non-null  int64         
 6   loss_sum           11160 non-null  float64       
 7   n_factual          11160 non-null  int64         
 8   n_possible         11160 non-null  int64         
 9   n_avoided          11160 non-null  int64         
 10  n_cons_activity    11160 non-null  int64         
 11  n_cons_reputation  11160 non-null  int64         
 12  n_cons_financial   11160 non-null  int64         
 13  risk_key           11160 non-null  object        
dtypes: dat

In [10]:
# dim_calendar = pd.read_csv('dim_calendar.csv', sep = ';')
# print(dim_calendar.shape)
# dim_calendar.head()

# Перенос данных в CH

In [11]:

import os
import sys 
sys.version

'3.7.6 | packaged by conda-forge | (default, Jun  1 2020, 18:57:50) \n[GCC 7.5.0]'

In [12]:
import getpass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [13]:
os.chdir('/home/jovyan/FR/superset')
os.getcwd()

'/home/jovyan/FR/superset'

In [14]:
jars = [r"hdfs:///LIB/COMMON/JAR/clickhouse-client-0.3.2-patch11.jar",
        r"hdfs:///LIB/COMMON/JAR/clickhouse-http-client-0.3.2-patch11.jar",
        r"hdfs:///LIB/COMMON/JAR/clickhouse-jdbc-0.3.2.jar"]

In [15]:
spark = SparkSession.builder.config('spark.hadoop.dfs.user.home.dir.prefix','/LAB/ASO/DVA_URVK/.USER').config('spark.jars', ','.join(jars)).appName('Spark_test').getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 10:06:19 WARN cluster.YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


2.4.0-cdh6.3.4


In [16]:
import os
os.getcwd()

'/home/jovyan/FR/superset'

In [17]:
sc = spark.sparkContext
sc.addFile('./CA.pem')

In [18]:
df1 = dashboard_measures.copy()
spark_df = spark.createDataFrame(df1)

In [19]:
spark_df.printSchema()

root
 |-- bp_number: long (nullable = true)
 |-- risk_name: string (nullable = true)
 |-- risk_key: string (nullable = true)
 |-- source_number: long (nullable = true)
 |-- source_label: string (nullable = true)
 |-- measure_type: string (nullable = true)
 |-- n_applied: long (nullable = true)
 |-- n_effective: long (nullable = true)
 |-- effectiveness: double (nullable = true)
 |-- effectiveness_level: string (nullable = true)
 |-- dominant_consequence_kind: string (nullable = true)
 |-- dominant_consequence_type: string (nullable = true)
 |-- report_month: timestamp (nullable = true)



In [20]:
import getpass 
user='SuvorovaAA@VIP.CBR.RU' 
#где user ввести свой логин IvanovII
passwd = 'AS!@#$as1234' 
# где password вести свой пароль 12345Abc, после выполнения всех ячеек удалите его, так как это общая зона и его смогут видеть все

#МЕНЯТЬ USER И PASSWORD НУЖНО ТОЛЬКО ЗДЕСЬ В ДРУГИХ ЯЧЕЙКАХ НИЧЕГО ТРОГАТЬ НЕ НАДО

In [21]:
driver="ru.yandex.clickhouse.ClickHouseDriver"
clickhouse_server = "clickhouse-aso-dva-urvk.apps.k8s.ehd.cbr.ru:443"
clickhouse_database = "zpe_aso_dva_urvk"
clickhouse_table = "dashboard_measures"
clickhouse_table_distributed = clickhouse_table + "_distributed"

In [22]:
clickhouse_url_ssl = f"jdbc:clickhouse://{clickhouse_server}/{clickhouse_database}?ssl=true&sslmode=strict&sslrootcert=./CA.pem"
print(clickhouse_url_ssl)

jdbc:clickhouse://clickhouse-aso-dva-urvk.apps.k8s.ehd.cbr.ru:443/zpe_aso_dva_urvk?ssl=true&sslmode=strict&sslrootcert=./CA.pem


In [23]:
properties = spark._sc._gateway.jvm.java.util.Properties()
properties.setProperty("driver", driver)
properties.setProperty("user", user)
properties.setProperty("password", passwd)

In [24]:
click_driver = sc._gateway.jvm.ru.yandex.clickhouse.ClickHouseDriver()

In [25]:
conn = click_driver.connect(clickhouse_url_ssl, properties)

In [26]:
statement = conn.createStatement()

In [27]:
spark.read.jdbc(url=clickhouse_url_ssl, table=f"{clickhouse_database}.{clickhouse_table_distributed}", properties=properties).count()

278

In [28]:
result = statement.execute(f"TRUNCATE TABLE {clickhouse_database}.{clickhouse_table} ON CLUSTER superset")

In [29]:
spark.read.jdbc(url=clickhouse_url_ssl, table=f"{clickhouse_database}.{clickhouse_table_distributed}", properties=properties).count()

0

In [30]:
import time

 # INSERT APPEND
# Запись DataFrame в ClickHouse, в распределённую по кластеру таблицу, метод APPEND
tsw = time.time_ns()
spark_df.write.mode("append") \
        .option('driver', properties['driver']) \
        .option('url', clickhouse_url_ssl) \
        .option('user', properties['user']) \
        .option('password', properties['password']) \
        .option("numPartitions", "30") \
        .option("batchsize","40000") \
        .jdbc(clickhouse_url_ssl, table = f"{clickhouse_database}.{clickhouse_table}")
tew = time.time_ns()
 
print(f"Elapsed time: {tew - tsw} ns")

[Stage 4:>                                                          (0 + 3) / 3]

Elapsed time: 2694227792 ns


In [31]:
spark_df.count()

1984

In [32]:
spark.stop()

# Код для создания dashboard_measures_cons

WITH aggregated AS (
  SELECT
    bp_number,
    risk_name, 
    year_month,
    max(n_factual) as n_factual,
    max(n_possible) as n_possible,
    max(n_avoided) as n_avoided,
    max(n_cons_activity) as n_cons_activity,
    max(n_cons_reputation) as n_cons_reputation,
    max(n_cons_financial) as n_cons_financial
  FROM dashboard_timeline
  GROUP BY bp_number, risk_name, year_month
  )

SELECT
  bp_number,
  risk_name, 
  year_month,
  'kind' as category_type,
  'Фактическое' as category_value,
  n_factual as val
FROM aggregated

UNION ALL
  
SELECT
  bp_number,
  risk_name, 
  year_month,
  'kind' as category_type,
  'Возможное' as category_value,
  n_possible as val
FROM aggregated   

UNION ALL
  
SELECT
  bp_number,
  risk_name, 
  toDate(year_month),
  'kind' as category_type,
  'Удалось избежать' as category_value,
  n_avoided as val
FROM aggregated 

UNION ALL
  
SELECT
  bp_number,
  risk_name, 
  year_month,
  'type' as category_type,
  'Деятельность' as category_value,
  n_cons_activity as val
FROM aggregated   

UNION ALL
  
SELECT
  bp_number,
  risk_name, 
  year_month,
  'type' as category_type,
  'Репутация' as category_value,
  n_cons_reputation as val
FROM aggregated  

UNION ALL
  
SELECT
  bp_number,
  risk_name, 
  year_month,
  'type' as category_type,
  'Финансы' as category_value,
  n_cons_financial as val
FROM aggregated 